# Capítulo 13 — Agrupamento no Mundo Real: Consenso, Restrições, Grafos e Fluxos

Códigos do capítulo. Execute as células em ordem; cada bloco é independente dos demais.


## 13.2.2 Receita em três passos

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score

X, y = make_moons(400, noise=0.08, random_state=0)

# 1) Gerar muitas partições diversas: K-means com k variado e subamostras de atributos/instâncias
rng = np.random.default_rng(0)
particoes = []
for _ in range(40):
    k = int(rng.integers(8, 16))                       # k grande de propósito: "super-segmentação"
    idx = rng.choice(len(X), int(0.8 * len(X)), replace=False)
    km = KMeans(k, n_init=1, random_state=int(rng.integers(10**6))).fit(X[idx])
    particoes.append(km.predict(X))

# 2) Matriz de coassociação: fração das partições em que i e j caem no mesmo grupo
n = len(X)
C = np.zeros((n, n))
for p in particoes:
    C += (p[:, None] == p[None, :])
C /= len(particoes)

# 3) Consenso: agrupamento hierárquico sobre a dissimilaridade 1 - C
cons = AgglomerativeClustering(n_clusters=2, metric="precomputed", linkage="average").fit_predict(1 - C)

print("ARI de cada partição individual (média):", np.mean([adjusted_rand_score(y, p) for p in particoes]).round(2))
print("ARI do K-means com k = 2              :", round(adjusted_rand_score(y, KMeans(2, n_init=10, random_state=0).fit_predict(X)), 2))
print("ARI do consenso                       :", round(adjusted_rand_score(y, cons), 2))

## 13.3.2 O COP-K-means

In [ ]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score

X, y = make_blobs(300, centers=3, cluster_std=2.4, random_state=3)
rng = np.random.default_rng(0)

# Restrições vindas de um especialista (simuladas a partir do gabarito): 20 pares
def sortear_restricoes(n_pares):
    ml, cl = [], []
    while len(ml) + len(cl) < n_pares:
        i, j = rng.choice(len(X), 2, replace=False)
        (ml if y[i] == y[j] else cl).append((int(i), int(j)))
    return ml, cl
ml, cl = sortear_restricoes(60)

def cop_kmeans(X, k, ml, cl, n_iter=50, seed=0):
    """K-means com restrições must-link / cannot-link (Wagstaff et al., 2001)."""
    r = np.random.default_rng(seed)
    # componentes must-link: fecho transitivo por união de conjuntos
    pai = list(range(len(X)))
    def raiz(a):
        while pai[a] != a:
            pai[a] = pai[pai[a]]; a = pai[a]
        return a
    for a, b in ml: pai[raiz(a)] = raiz(b)
    comp = np.array([raiz(i) for i in range(len(X))])
    cl_por_ponto = {}
    for a, b in cl:
        cl_por_ponto.setdefault(a, set()).add(b); cl_por_ponto.setdefault(b, set()).add(a)
    centros = X[r.choice(len(X), k, replace=False)]
    rot = -np.ones(len(X), int)
    for _ in range(n_iter):
        novo = -np.ones(len(X), int)
        for i in r.permutation(len(X)):
            if novo[i] >= 0: continue
            membros = np.where(comp == comp[i])[0]       # o bloco must-link inteiro anda junto
            d = ((X[membros][:, None] - centros[None]) ** 2).sum(2).sum(0)
            for c in np.argsort(d):
                proibido = any(novo[v] == c for m in membros for v in cl_por_ponto.get(m, ()))
                if not proibido:
                    novo[membros] = c; break
            else:
                return None                                # restrições inviáveis para esta inicialização
        if (novo == rot).all(): break
        rot = novo
        centros = np.array([X[rot == c].mean(0) if (rot == c).any() else centros[c] for c in range(k)])
    return rot

def violacoes(rot):
    return sum(rot[a] != rot[b] for a, b in ml) + sum(rot[a] == rot[b] for a, b in cl)

from sklearn.cluster import KMeans
livre = KMeans(3, n_init=10, random_state=0).fit_predict(X)
melhor = None
for s in range(10):
    r = cop_kmeans(X, 3, ml, cl, seed=s)
    if r is not None:
        sse = sum(((X[r == c] - X[r == c].mean(0)) ** 2).sum() for c in range(3))
        if melhor is None or sse < melhor[0]: melhor = (sse, r)
rest = melhor[1]
print("K-means livre:       ARI =", round(adjusted_rand_score(y, livre), 2), "| restrições violadas:", violacoes(livre), "de", len(ml) + len(cl))
print("K-means restrito:    ARI =", round(adjusted_rand_score(y, rest), 2), "| restrições violadas:", violacoes(rest), "de", len(ml) + len(cl))

## 13.4.3 Um laboratório comparativo

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import adjusted_rand_score

# Clientes sintéticos com 3 perfis ocultos; atributos numéricos, categóricos e ordinal
rng = np.random.default_rng(0)
n = 450
perfil = rng.integers(0, 3, n)
canais = [["app", "whatsapp"], ["loja", "telefone"], ["site", "email"]]   # cada perfil usa 2 canais típicos
todos = ["app", "whatsapp", "loja", "telefone", "site", "email"]
canal = [rng.choice(canais[p]) if rng.random() < 0.8 else rng.choice(todos) for p in perfil]
plano = [rng.choice(["basico", "padrao", "premium"], p=pr) for pr in
         np.array([[0.6, 0.3, 0.1], [0.1, 0.3, 0.6], [0.2, 0.6, 0.2]])[perfil]]
df = pd.DataFrame({
    "idade": rng.normal(np.array([32, 42, 52])[perfil], 11),        # perfis com sobreposição forte
    "renda": rng.normal(np.array([3.5, 5.0, 4.0])[perfil], 1.6),
    "canal": canal,
    "plano": plano,
})
num, cat, ordi = ["idade", "renda"], ["canal"], ["plano"]
df["plano_ord"] = df["plano"].map({"basico": 0, "padrao": 1, "premium": 2})
numord = num + ["plano_ord"]

# 1) Ingênua: códigos numéricos arbitrários para as categorias
ing = df[numord].copy(); ing["canal"] = df["canal"].astype("category").cat.codes
r1 = KMeans(3, n_init=10, random_state=0).fit_predict(StandardScaler().fit_transform(ing))

# 2) One-hot + padronização + K-means
Xn = StandardScaler().fit_transform(df[numord])
X2 = np.hstack([Xn, OneHotEncoder(sparse_output=False).fit_transform(df[cat])])
r2 = KMeans(3, n_init=10, random_state=0).fit_predict(X2)

# 3) Gower (média das diferenças parciais) + hierárquico de ligação média
def gower(df, numericos, categoricos):
    D = np.zeros((len(df), len(df)))
    for c in numericos:                                    # |diferença| / amplitude
        v = df[c].to_numpy(float); D += np.abs(v[:, None] - v[None, :]) / (v.max() - v.min())
    for c in categoricos:                                  # 0 se igual, 1 se diferente
        v = df[c].to_numpy(); D += (v[:, None] != v[None, :])
    return D / (len(numericos) + len(categoricos))
G = gower(df, numord, cat)
r3 = AgglomerativeClustering(3, metric="precomputed", linkage="average").fit_predict(G)

# 4) K-prototypes simples: média nos numéricos, moda nos categóricos; custo = sq. euclidiana + gamma * discordâncias
def kprototypes(Xn, Xc, k, gamma, seed, n_iter=50):
    r = np.random.default_rng(seed)
    ini = r.choice(len(Xn), k, replace=False)
    cn, cc = Xn[ini].copy(), Xc[ini].copy()
    rot = -np.ones(len(Xn), int)
    for _ in range(n_iter):
        d = ((Xn[:, None] - cn[None]) ** 2).sum(2) + gamma * (Xc[:, None, :] != cc[None]).sum(2)
        novo = d.argmin(1)
        if (novo == rot).all(): break
        rot = novo
        for j in range(k):
            m = rot == j
            if m.any():
                cn[j] = Xn[m].mean(0); cc[j] = [pd.Series(Xc[m, c]).mode()[0] for c in range(Xc.shape[1])]
    custo = d[np.arange(len(Xn)), rot].sum()
    return rot, custo

Xc = df[cat].to_numpy()
r4 = min((kprototypes(Xn, Xc, 3, gamma=1.5, seed=s) for s in range(10)), key=lambda t: t[1])[0]

for nome, r in [("Códigos numéricos arbitrários", r1), ("One-hot + K-means", r2), ("Gower + hierárquico", r3), ("K-prototypes", r4)]:
    print(f"{nome:32s} ARI = {adjusted_rand_score(perfil, r):.2f}")

## 13.5.2 Os métodos do scikit-learn

In [ ]:
import numpy as np
from sklearn.datasets import make_checkerboard
from sklearn.cluster import SpectralBiclustering
from sklearn.metrics import consensus_score

# Matriz usuários × itens com 4 "blocos" (checkerboard) escondidos pelo embaralhamento
dados, linhas, colunas = make_checkerboard(shape=(300, 200), n_clusters=(4, 3), noise=8, shuffle=False, random_state=0)
rng = np.random.default_rng(0)
pl, pc = rng.permutation(300), rng.permutation(200)
embaralhada = dados[pl][:, pc]                              # o que veríamos na prática

bc = SpectralBiclustering(n_clusters=(4, 3), method="log", random_state=0).fit(embaralhada)

# Para comparar com a verdade, aplicamos a mesma permutação aos rótulos verdadeiros
print("Consenso (biclustering espectral, xadrez):", round(consensus_score(bc.biclusters_, (linhas[:, pl], colunas[:, pc])), 2))
print("Linhas por grupo de linhas :", np.bincount(bc.row_labels_))
print("Colunas por grupo de colunas:", np.bincount(bc.column_labels_))

# Reordenar a matriz pelos rótulos revela os blocos
ordem_l, ordem_c = np.argsort(bc.row_labels_), np.argsort(bc.column_labels_)
reordenada = embaralhada[ordem_l][:, ordem_c]
print("Formato da matriz reordenada:", reordenada.shape)

## 13.6.3 Louvain e Leiden

In [ ]:
import networkx as nx
from networkx.algorithms.community import louvain_communities, modularity
from sklearn.metrics import adjusted_rand_score

G = nx.karate_club_graph()                                  # rede de 34 pessoas (Zachary, 1977)
real = [G.nodes[i]["club"] for i in G.nodes]                 # "Mr. Hi" ou "Officer": a divisão observada

for res in (0.5, 1.0, 2.0):
    com = louvain_communities(G, resolution=res, seed=0)
    rot = {v: i for i, c in enumerate(com) for v in c}
    pred = [rot[v] for v in G.nodes]
    print(f"resolução {res}: {len(com)} comunidades | modularidade Q = {modularity(G, com):.3f} | ARI vs. clube = {adjusted_rand_score(real, pred):.2f}")

## 13.7.3 Simulando a deriva

In [ ]:
import numpy as np
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import adjusted_rand_score

rng = np.random.default_rng(0)
base = np.array([[0.0, 0.0], [6.0, 0.0], [3.0, 5.0]])

def lote(t, n=300):
    """Lote no instante t: os centros derivam com o tempo e o grupo 2 só aparece a partir de t = 6."""
    centros = base + np.array([[0.3 * t, 0.0], [0.0, 0.25 * t], [-0.2 * t, 0.0]])
    ativos = [0, 1] if t < 6 else [0, 1, 2]
    y = rng.choice(ativos, n)
    return centros[y] + rng.normal(0, 0.8, (n, 2)), y

lotes = [lote(t) for t in range(12)]

congelado = KMeans(3, n_init=10, random_state=0).fit(lotes[0][0])           # nunca atualiza
incremental = MiniBatchKMeans(3, n_init=3, random_state=0)                  # partial_fit: acumula tudo
janela = []                                                                  # janela deslizante: só os 3 últimos lotes

print(" t | ARI congelado | ARI incremental | ARI janela | deslocamento do centro (janela)")
centros_ant = None
for t, (X, y) in enumerate(lotes):
    incremental.partial_fit(X)
    janela.append(X); janela = janela[-3:]
    modelo_j = KMeans(3, n_init=5, random_state=0).fit(np.vstack(janela))
    desloc = np.nan
    if centros_ant is not None:
        D = np.linalg.norm(modelo_j.cluster_centers_[:, None] - centros_ant[None], axis=2)
        desloc = D.min(1).mean()
    centros_ant = modelo_j.cluster_centers_
    if t >= 1:
        print(f"{t:2d} | {adjusted_rand_score(y, congelado.predict(X)):13.2f} | {adjusted_rand_score(y, incremental.predict(X)):15.2f} | {adjusted_rand_score(y, modelo_j.predict(X)):10.2f} | {desloc:.2f}")

## 13.8.2 Um laboratório com um corpus sintético

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import NMF, LatentDirichletAllocation, TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.preprocessing import Normalizer
from sklearn.metrics import adjusted_rand_score

# Corpus sintético em português: 3 assuntos + palavras comuns a todos
rng = np.random.default_rng(0)
vocab = {
    "esporte": "gol time jogo campeonato técnico torcida estádio bola atacante goleiro".split(),
    "saude":   "médico hospital vacina paciente tratamento sintoma remédio doença exame consulta".split(),
    "economia": "inflação juros mercado dólar banco investimento imposto bolsa crédito salário".split(),
}
comuns = "hoje ontem governo cidade semana novo grande segundo mês parte".split()
docs, assunto = [], []
for _ in range(300):
    a = rng.choice(list(vocab)); assunto.append(a)
    palavras = list(rng.choice(vocab[a], 5)) + list(rng.choice(comuns, 5)) + list(rng.choice(sum(vocab.values(), []), 5))
    rng.shuffle(palavras); docs.append(" ".join(palavras))

cv = CountVectorizer(); Xc = cv.fit_transform(docs)
tf = TfidfVectorizer(); Xt = tf.fit_transform(docs)
termos_c, termos_t = cv.get_feature_names_out(), tf.get_feature_names_out()

def mostrar(nome, comp, termos):
    print(nome)
    for k, c in enumerate(comp):
        print(f"  tópico {k}:", ", ".join(termos[np.argsort(c)[::-1][:5]]))

nmf = NMF(3, init="nndsvda", random_state=0, max_iter=500).fit(Xt)
lda = LatentDirichletAllocation(3, random_state=0, max_iter=30).fit(Xc)
mostrar("NMF (TF-IDF)", nmf.components_, termos_t)
mostrar("LDA (contagens)", lda.components_, termos_c)

# Agrupamento sobre uma representação densa (aqui, LSA: SVD do TF-IDF, normalizada)
Z = Normalizer().fit_transform(TruncatedSVD(10, random_state=0).fit_transform(Xt))
rot = KMeans(3, n_init=10, random_state=0).fit_predict(Z)
print("ARI NMF :", round(adjusted_rand_score(assunto, nmf.transform(Xt).argmax(1)), 2))
print("ARI LDA :", round(adjusted_rand_score(assunto, lda.transform(Xc).argmax(1)), 2))
print("ARI K-means sobre LSA:", round(adjusted_rand_score(assunto, rot), 2))

# Coerência simples (UMass): co-ocorrência dos termos principais nos documentos
B = (Xc > 0).astype(int).toarray()
def umass(comp, termos, topn=5):
    vals = []
    for c in comp:
        idx = [list(termos_c).index(t) for t in termos[np.argsort(c)[::-1][:topn]]]
        s = 0
        for i in range(1, len(idx)):
            for j in range(i):
                s += np.log((B[:, idx[i]] * B[:, idx[j]]).sum() + 1) - np.log(B[:, idx[j]].sum())
        vals.append(s)
    return np.mean(vals)
print("Coerência UMass  NMF:", round(umass(nmf.components_, termos_t), 2), "| LDA:", round(umass(lda.components_, termos_c), 2))

## 13.9.2 Três ferramentas

In [ ]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import adjusted_rand_score

X, y = make_blobs(900, centers=4, cluster_std=1.0, random_state=7)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

# --- 1) Atribuição de novos casos com um modelo que tem predict()
km = KMeans(4, n_init=10, random_state=0).fit(X_tr)
print("ARI no treino:", round(adjusted_rand_score(y_tr, km.labels_), 3), "| ARI em casos novos:", round(adjusted_rand_score(y_te, km.predict(X_te)), 3))

# --- 2) Rejeição: o caso novo está longe de qualquer grupo conhecido?
d_tr = km.transform(X_tr).min(1)
limite = np.quantile(d_tr, 0.99)                       # 99º percentil das distâncias do treino ao centro
novos = np.vstack([X_te, [[15, 15], [-15, 12]]])      # dois casos de um "grupo desconhecido"
d_novos = km.transform(novos).min(1)
rejeitados = d_novos > limite
print("Rejeitados entre os casos normais:", int(rejeitados[:len(X_te)].sum()), "de", len(X_te), "| entre os desconhecidos:", int(rejeitados[-2:].sum()), "de 2")

# --- 3) Métodos sem predict() (DBSCAN, hierárquico, espectral): treinar um classificador sobre os rótulos
db = DBSCAN(eps=0.9, min_samples=6).fit(X_tr)
dentro = db.labels_ >= 0
knn = KNeighborsClassifier(15).fit(X_tr[dentro], db.labels_[dentro])
print("DBSCAN + kNN em casos novos: ARI =", round(adjusted_rand_score(y_te, knn.predict(X_te)), 3))

# --- 4) Força de predição (Tibshirani; Walther, 2005): o agrupamento aprendido no treino vale no teste?
def forca_predicao(k, seed=0):
    m_tr = KMeans(k, n_init=10, random_state=seed).fit(X_tr)
    m_te = KMeans(k, n_init=10, random_state=seed).fit(X_te)      # "verdade" no teste: agrupa o teste sozinho
    prev = m_tr.predict(X_te)                                     # previsão do modelo de treino para o teste
    ps = []
    for j in range(k):
        idx = np.where(m_te.labels_ == j)[0]
        if len(idx) < 2: continue
        iguais = prev[idx][:, None] == prev[idx][None, :]
        ps.append((iguais.sum() - len(idx)) / (len(idx) * (len(idx) - 1)))   # pares que continuam juntos
    return min(ps)

for k in range(2, 8):
    print(f"k = {k}: força de predição = {forca_predicao(k):.2f}")

## Resumo do capítulo